In [8]:
%pip install fastapi uvicorn pydantic

     ---------------------------------------- 0.0/95.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/95.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/95.6 kB ? eta -:--:--
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ------------ --------------------------- 30.7/95.6 kB 1.3 MB/s eta 0:00:01
     ---------------- ---------------------- 41.0/95.6 kB 78.6 kB/s eta 0:00:01
     ---------------- ---------------------- 41.0/95.6 kB 78.6 k


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [61]:
import io
import matplotlib.pyplot as plt
import uvicorn
import numpy as np
import logging
from pathlib import Path
from typing import List
from joblib import load
import nest_asyncio
from enum import Enum
from fastapi import FastAPI, HTTPException, Depends
from fastapi.responses import StreamingResponse
from pydantic import BaseModel,  conlist, Field
from sklearn.pipeline import Pipeline

2025-08-24 20:14:44,804 DEBUG matplotlib matplotlib data path: d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\matplotlib\mpl-data
2025-08-24 20:14:45,050 DEBUG matplotlib CONFIGDIR=C:\Users\crist\.matplotlib
2025-08-24 20:14:45,057 DEBUG matplotlib interactive is False
2025-08-24 20:14:45,060 DEBUG matplotlib platform is win32
2025-08-24 20:14:46,334 DEBUG matplotlib CACHEDIR=C:\Users\crist\.matplotlib
2025-08-24 20:14:46,355 DEBUG matplotlib.font_manager Using fontManager instance from C:\Users\crist\.matplotlib\fontlist-v330.json


In [ ]:
# Directorio donde están tus modelos
MODELS_DIR = Path.cwd().parent  / "models"

# Diccionario en memoria que mapea k -> pipeline cargado
pipelines: dict[int, any] = {}

In [40]:
from fastapi.responses import JSONResponse

@app.exception_handler(Exception)
async def all_exception_handler(request, exc):
    logger.exception("Unhandled exception:")
    return JSONResponse(
        status_code=500,
        content={"detail": str(exc)}
    )


In [ ]:

# Interactuamos con la API usando este elemento
app = FastAPI(title='Implementando un modelo de Machine Learning')

# Logger
logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

# -------------------------------------------------------------------------------------------
# Carga dinámica de todos los pipelines 
# Cargar todos los joblib que tienen el mismo patrón
for path in MODELS_DIR.glob("pipeline_kmeans*.joblib"):
    # Extraemos el número de k del nombre del archivo
    stem = path.stem                    # e.g. "pipeline_kmeans3"
    k = int(stem.replace("pipeline_kmeans", ""))  
    pipelines[k] = load(path)
logger.info("Pipelines cargados: %s", list(pipelines.keys()))

# ------------------------------------------------------------------------------------------
# Modelos de request/response con Pydantic
# Definimos las clases de request/response

class PredictRequest(BaseModel):
    k: int = Field(gt=0, description="Número de clústeres (k) deseado")
    data: list[list[float]] = Field(..., description="Datos a clasificar (lista de observaciones)")


"""
class PredictRequest(BaseModel):
    k: int
    data: List[conlist(float, min_items=2, max_items=2)]
"""
class PredictResponse(BaseModel):
    k: int
    clusters: list[int]
    inertia: float
    
#--------------------------------------------------------------------------------------

# Dependencia para fetch de pipeline
def get_pipeline_or_404(k: int) -> Pipeline:
    pipe = pipelines.get(k)
    if not pipe:
        raise HTTPException(404, f"No existe modelo para k={k}")
    return pipe

# -------------------------------------------------------------------------------------------
# Defenimos un método GET para el endpoint
@app.get("/")
def home():
    return "¡Felicitaciones!, tu API está funcionando según lo esperado. Anda ahora a http://localhost:8000/docs."


# -------------------------------------------------------------------------------------------
# Endpoint de debug para inspeccionar pipelines
@app.get("/debug/pipelines")
def debug_pipelines():
    """
    Devuelve un dict con:
      k -> lista de nombres de pasos en el pipeline
    """
    return {
        k: list(pipe.named_steps.keys())
        for k, pipe in pipelines.items()
    }

# ------------------------------------------------------------------------------------

# Este endpoint maneja la lógica necesaria para clasificar.
# Requiere como entrada el vector de características del viaje y el umbral de confianza para la clasificación.
# Crear el endpoint de predicción
@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest, pipeline: Pipeline = Depends(get_pipeline_or_404)):
    # Log de entrada
    logger.debug("Received /predict request — k: %s, data length: %d", req.k, len(req.data))
    
    # Validación de forma
    X = np.array(req.data)
    if X.ndim != 2:
        raise HTTPException(422, "La entrada debe ser una matriz 2D")
    expected = pipeline.named_steps["kmeans"].n_features_in_
    if X.shape[1] != expected:
        raise HTTPException(422, f"Se esperaban {expected} features, recibidas {X.shape[1]}")

    # Predicción y manejo de errores
    try:
        clusters = pipeline.predict(X)
    except NotFittedError:
        logger.error("Pipeline no entrenado para k=%s", req.k, exc_info=True)
        raise HTTPException(500, "Modelo no listo para predecir")
    except Exception:
        logger.exception("Error inesperado en /predict")
        raise HTTPException(500, "Error interno en el servidor")

    inertia = float(pipeline.named_steps["kmeans"].inertia_)
    return PredictResponse(k=req.k, clusters=clusters.tolist(), inertia=inertia)

2025-08-24 19:22:13,357 INFO __main__ Pipelines cargados: [1, 2, 3, 4, 5, 6, 7]


In [62]:
@app.get("/plot")
def plot_clusters(k:int, data: list[list[float]]):
    if k not in pipelines:
        raise HTTPException(status_code=404, detail=f"Modelo para k={k} no encontrado")

    
    X = mp.array(data)
    labels = pipelines[k].predict(X)
    centroids = pipelines[k].cluster_centers_
    
    # Visualizar
    fig, ax = plt.subplots()
    scatter = ax.scatter(X[:,0], X[:,1], c=labels, cmap="tab10", s=50)
    ax.scatter(centroids[:,0], centroids[:,1], c="black", s=200, marker="X")
    ax.set_title(f"Agrupación k={k}")
    
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    
    return StreamingResponse(buf, media_type="image/png")

¡Corriendo la celda que viene echaremos a andar el servidor!

Esto causará que el notebook se bloquee (No podremos correr más celdas) hasta que interrumpamos de forma manual el kernel.
Podemos hacer eso haciendo click en la pestaña **kernel** y luego **Interrupt**.

In [60]:
logging.basicConfig(level=logging.DEBUG)

# Esto deja correr al servidor en un ambiente interactivo como un jupyter notebook
nest_asyncio.apply()

# Donde se hospedará el servidor
host = "127.0.0.1"

# Iniciamos el servidor
uvicorn.run(app, host= host, port=8000)

INFO:     Started server process [9876]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:53244 - "GET / HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9876]


¡El servidor está corriendo! Vamos a http://localhost:8000/ para verlo en acción.

In [38]:
import joblib
import numpy as np

pipe = joblib.load(Path.cwd().parent  / "models/pipeline_kmeans3.joblib")
X = np.array([[1.24, -0.47], [-0.85, 2.13]])

# ¿Falla aquí?
clusters = pipe.predict(X)
print("Clusters:", clusters)
print("Centers shape:", pipe.named_steps["Kmeans"].cluster_centers_.shape)

Clusters: [0 2]
Centers shape: (3, 2)


In [46]:
import requests

resp = requests.post("http://127.0.0.1:8000/predict", json=payload)
print("Status:", resp.status_code)
print("Body  :", resp.text)





2025-08-24 18:36:27,423 ERROR asyncio Task exception was never retrieved
future: <Task finished name='Task-104' coro=<Server.serve() done, defined at d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\main.py", line 580, in run
    server.run()
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionCli

NameError: name 'payload' is not defined